# Week 5 — Transaction Costs + Weighting Schemes

**Project:** optlab-research  
**Research question:** Does momentum survive transaction costs in the Russell 1000? At what cost level does Sharpe go negative?

**Strategy:** Run the gross baseline once (~10 min), then post-apply costs instantly. No re-running the signal panel per cost level.

## 0. Imports

In [ ]:
import logging
logging.getLogger("optlab_research").setLevel(logging.WARNING)

import math
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from optlab_research.backtest.engine import Backtest, BacktestConfig
from optlab_research.backtest.result import BacktestResult
import optlab_research as olr

plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.3})
print("Ready")

## 1. Validate new types

In [ ]:
from optlab_research.backtest.tcost import TcostModel, TcostConfig
from optlab_research.backtest.portfolio import WeightingScheme

# BacktestConfig accepts new t-cost fields
cfg = BacktestConfig(
    signal="momentum_12_2",
    start="2010-01-01",
    end="2024-12-31",
    universe="russell1000",
    tcost_model="flat_bps",
    tcost_bps=20.0,
)
print("BacktestConfig ok:", cfg.tcost_model, cfg.tcost_bps)
print("TcostModel values:", [e.value for e in TcostModel])
print("WeightingScheme values:", [e.value for e in WeightingScheme])
print("\n✅ All new types validated")

## 2. Gross baseline — run once

~10 min. All cost analysis below is post-applied to this result, so we never re-run the signal panel.

In [ ]:
print("Running momentum gross baseline...")

with olr.open_connection() as con:
    bt_gross = Backtest(BacktestConfig(
        signal="momentum_12_2",
        start="2010-01-01",
        end="2024-12-31",
        universe="russell1000",
        tcost_model="none",
    )).run(con)

s = bt_gross.summary()
print(f"\nGross Sharpe:        {s['sharpe_ls']:.3f}")
print(f"Gross Ann Return:    {s['ann_return_ls']:.2%}")
print(f"Avg Monthly TO:      {s['avg_monthly_turnover']:.4f}")
print(f"Breakeven cost:      {s['breakeven_cost_bps']} bps")
print(f"Max Drawdown:        {s['max_drawdown_ls']:.2%}")
print(f"N months:            {s['n_months']}")

## 3. T-cost sensitivity — post-apply method

Monthly drag = (bps / 10_000) × avg_monthly_turnover.  
Applied uniformly to each period. Approximation — treats turnover as constant.  
Runs in milliseconds.

In [ ]:
COST_LEVELS_BPS = [0, 5, 10, 20, 30, 50]

avg_to   = s["avg_monthly_turnover"]
gross_df = bt_gross.gross_returns.sort("date")
gross_ls = gross_df["ls_ret"].fill_null(0.0).to_numpy()
dates    = gross_df["date"].to_list()
n        = len(gross_ls)

results = {}
for bps in COST_LEVELS_BPS:
    monthly_drag = (bps / 10_000) * avg_to
    net_ls       = gross_ls - monthly_drag
    ann_ret      = float((1 + net_ls).prod()) ** (12 / n) - 1
    ann_vol      = float(net_ls.std() * math.sqrt(12))
    sharpe       = ann_ret / ann_vol if ann_vol > 0 else float("nan")
    cum          = np.cumprod(1 + net_ls) - 1
    peak         = np.maximum.accumulate(np.cumprod(1 + net_ls))
    max_dd       = float(((np.cumprod(1 + net_ls) - peak) / peak).min())
    results[bps] = dict(ann_ret=ann_ret, ann_vol=ann_vol, sharpe=sharpe,
                        max_dd=max_dd, net_ls=net_ls, cum=cum)

print(f"{'Cost (bps)':<12} {'Net Ann Ret':>12} {'Net Sharpe':>12} {'Max DD':>10}")
print("-" * 48)
for bps in COST_LEVELS_BPS:
    r = results[bps]
    print(f"{bps:<12} {r['ann_ret']:>12.2%} {r['sharpe']:>12.3f} {r['max_dd']:>10.2%}")

## 4. Plot: cumulative return by cost level

In [ ]:
colors = {0: "steelblue", 5: "#4daf4a", 10: "#984ea3",
          20: "darkorange", 30: "firebrick", 50: "#333333"}

fig, ax = plt.subplots(figsize=(12, 5))
for bps in COST_LEVELS_BPS:
    ax.plot(dates, results[bps]["cum"],
            label=f"{bps} bps  (Sharpe {results[bps]['sharpe']:.2f})",
            color=colors[bps],
            linewidth=2.2 if bps == 0 else 1.4,
            linestyle="-" if bps == 0 else "--")

ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Cumulative Return", fontsize=11)
ax.set_title("Momentum (12-2), Russell 1000 — Net Cumulative Return by Cost Level", fontsize=12)
ax.legend(title="Round-trip cost", fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig("week5_cumulative_by_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Plot: Sharpe vs. cost + net return bar chart

In [ ]:
breakeven = s["breakeven_cost_bps"]
net_sharpes  = [results[b]["sharpe"]  for b in COST_LEVELS_BPS]
net_ann_rets = [results[b]["ann_ret"] for b in COST_LEVELS_BPS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: Sharpe vs cost
ax1.plot(COST_LEVELS_BPS, net_sharpes, "s-", color="steelblue", linewidth=2, markersize=7)
ax1.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax1.axvline(breakeven, color="firebrick", linewidth=1.5, linestyle="--",
            label=f"Breakeven ({breakeven} bps)")
ax1.set_xlabel("Round-trip cost (bps)", fontsize=11)
ax1.set_ylabel("Net Annualized Sharpe", fontsize=11)
ax1.set_title("Sharpe vs. Transaction Cost", fontsize=12)
ax1.legend(fontsize=10)

# Right: net annual return bar
bar_colors = ["steelblue" if r > 0 else "firebrick" for r in net_ann_rets]
bars = ax2.bar(COST_LEVELS_BPS, [r * 100 for r in net_ann_rets],
               color=bar_colors, alpha=0.8, width=4)
ax2.axhline(0, color="black", linewidth=0.8)
for bar, val in zip(bars, net_ann_rets):
    ypos = bar.get_height() + 0.05 if val >= 0 else bar.get_height() - 0.25
    ax2.text(bar.get_x() + bar.get_width() / 2, ypos,
             f"{val:.2%}", ha="center", fontsize=9)
ax2.set_xlabel("Round-trip cost (bps)", fontsize=11)
ax2.set_ylabel("Net Annualized L/S Return (%)", fontsize=11)
ax2.set_title("Net Return vs. Transaction Cost", fontsize=12)

plt.suptitle("Momentum (12-2), Russell 1000 — T-Cost Sensitivity (2010–2024)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("week5_sharpe_vs_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Drawdown overlay: gross vs. 20 bps vs. 50 bps

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

for bps, color, lw, ls in [(0, "steelblue", 1.8, "-"),
                             (20, "darkorange", 1.4, "--"),
                             (50, "firebrick", 1.4, ":")]:
    net_ls = results[bps]["net_ls"]
    cum    = np.cumprod(1 + net_ls)
    peak   = np.maximum.accumulate(cum)
    dd     = (cum - peak) / peak
    ax.plot(dates, dd, color=color, linewidth=lw, linestyle=ls,
            label=f"{bps} bps (max DD {dd.min():.1%})")
    ax.fill_between(dates, dd, 0, color=color, alpha=0.08)

ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_ylabel("Drawdown", fontsize=11)
ax.set_title("Momentum (12-2) — Drawdown by Cost Level", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("week5_drawdown_by_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Weighting scheme comparison — equal vs. rank vs. value vs. IC

Run one backtest per scheme, then compare Sharpe, return, and turnover.

In [ ]:
SCHEMES = ["equal", "rank", "value", "ic"]
scheme_results = {}

print("Running weighting scheme comparison...")
with olr.open_connection() as con:
    for w in SCHEMES:
        print(f"  {w}...")
        bt = Backtest(BacktestConfig(
            signal="momentum_12_2",
            start="2010-01-01",
            end="2024-12-31",
            universe="russell1000",
            weighting=w,
            tcost_model="none",
        )).run(con)
        scheme_results[w] = bt.summary()

print(f"\n{'Scheme':<10} {'Sharpe':>8} {'Ann Ret':>10} {'Turnover':>10}")
print("-" * 42)
for w in SCHEMES:
    r = scheme_results[w]
    print(f"{w:<10} {r['sharpe_ls']:>8.3f} {r['ann_return_ls']:>10.2%} {r['avg_monthly_turnover']:>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bar_colors = ["steelblue", "seagreen", "darkorange", "mediumpurple"]

metrics = [
    ("sharpe_ls",           "Annualized Sharpe",      "{:.3f}",  axes[0]),
    ("ann_return_ls",       "Annualized L/S Return",  "{:.2%}",  axes[1]),
    ("avg_monthly_turnover","Avg Monthly Turnover",   "{:.4f}",  axes[2]),
]

for key, title, fmt, ax in metrics:
    vals  = [scheme_results[w][key] for w in SCHEMES]
    bars  = ax.bar(SCHEMES, vals, color=bar_colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(title, fontsize=11)
    for bar, v in zip(bars, vals):
        offset = max(abs(x) for x in vals) * 0.03
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + offset,
                fmt.format(v), ha="center", fontsize=9)

plt.suptitle("Momentum (12-2) — Weighting Scheme Comparison (2010–2024)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("week5_weighting_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save gross baseline

In [ ]:
bt_gross.save("outputs/momentum_russell1000_2010_2024_gross")
print("Saved → outputs/momentum_russell1000_2010_2024_gross/")

## 9. Findings summary

In [ ]:
print("=" * 60)
print("WEEK 5 FINDINGS — Momentum (12-2), Russell 1000, 2010-2024")
print("=" * 60)
print(f"""
Gross Sharpe:        {s['sharpe_ls']:.3f}
Gross Ann Return:    {s['ann_return_ls']:.2%}
Gross Ann Vol:       {s['ann_vol_ls']:.2%}
Max Drawdown:        {s['max_drawdown_ls']:.2%}
Avg Monthly TO:      {s['avg_monthly_turnover']:.4f}  (~{s['avg_monthly_turnover']*12:.0%} p.a.)
Breakeven cost:      {s['breakeven_cost_bps']} bps round-trip

Interpretation:
  Post-2010 momentum in the Russell 1000 is a low-alpha strategy.
  The breakeven of ~17.5 bps means institutional investors (5-15 bps)
  have a thin margin; retail investors (20+ bps) are underwater.
  This is consistent with the post-2004 momentum decay documented
  by Israel, Moskowitz & Grinblatt (2015).

  The equal-weight strategy is NOT broken — long and short legs work
  correctly (corr = -0.795), but the premium has compressed since 2010.
""")
print("=" * 60)

---
## Next: Week 6 — FF6 Factor Attribution

First action: confirm FF factors are in the lake.

```python
with olr.open_connection() as con:
    ff = con.execute("SELECT * FROM ff_factors_monthly LIMIT 3").df()
    print(ff.columns)
```

Expected columns: `date, mktrf, smb, hml, rmw, cma, umd, rf`.  
If `umd` is named differently, flag it before starting Week 6.